# scVI Integration (HVG) - Myeloid Compartment

Trains scVI on the myeloid compartment restricted to highly variable genes (HVGs),
then saves both the latent embeddings and the integrated AnnData object for
downstream clustering.

In [ ]:
# !pip install --quiet scanpy scvi-tools==1.2.1 scikit-misc  # scikit-misc is required by the seurat_v3 HVG flavor

In [ ]:
import os

import numpy as np
import pandas as pd
import scanpy as sc
import scvi

## Config

In [ ]:
input_path = "sub_Myeloid_raw.h5ad"

output_dir = "./scvi_results"
output_path = "sub_Myeloid_raw_scvi_combine.h5ad"

n_top_hvgs = 3000
hvg_batch_key = None  # pooled HVGs: some samples have too few myeloid cells for per-batch seurat_v3

batch_key = "ID"
categorical_covariate_keys = ["batchseq"]
continuous_covariate_keys = ["percent_mito"]
n_latent_values = [10, 20, 30]
latent_key_prefix = "X_scVI_n_latent_"  # key used by downstream myeloid notebooks
min_cells_per_batch = 3                 # samples below this are reported (scVI warns below 3 cells)

os.makedirs(output_dir, exist_ok=True)
scvi.settings.seed = 0
print("scvi-tools version:", scvi.__version__)

## Load and preprocess data

In [ ]:
adata = sc.read(input_path)  # non-normalized raw counts
adata.layers["raw_counts"] = adata.X.copy()  # preserve raw counts before normalizing

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

batch_sizes = adata.obs[batch_key].value_counts()
small = batch_sizes[batch_sizes < min_cells_per_batch]
if len(small):
    print(f"samples with < {min_cells_per_batch} cells: {small.to_dict()}")
adata

## Identify highly variable genes and subset

In [ ]:
sc.pp.highly_variable_genes(
    adata, n_top_genes=n_top_hvgs, flavor="seurat_v3",
    subset=False, batch_key=hvg_batch_key, layer="raw_counts",
)

hvg_mask = adata.var["highly_variable"].tolist()
bdata = adata[:, hvg_mask].copy()  # carries raw_counts layer subset automatically
bdata

## Train scVI on the HVG subset

In [ ]:
scvi.model.SCVI.setup_anndata(
    bdata,
    layer="raw_counts",
    batch_key=batch_key,
    categorical_covariate_keys=categorical_covariate_keys,
    continuous_covariate_keys=continuous_covariate_keys,
)

models = {}
for n_latent in n_latent_values:
    print("Training scVI model with n_latent:", n_latent)
    models[n_latent] = scvi.model.SCVI(bdata, n_latent=n_latent)
    models[n_latent].train()

## Extract and save latent embeddings

In [ ]:
for n_latent in n_latent_values:
    latent_key = f"{latent_key_prefix}{n_latent}"
    # model was trained on bdata, but embeddings are stored back onto the full
    # adata (same cells, same order) so downstream analyses keep all genes
    adata.obsm[latent_key] = models[n_latent].get_latent_representation()

    latent_df = pd.DataFrame(adata.obsm[latent_key], index=adata.obs.index)
    out_path = os.path.join(output_dir, f"hvg_obsm_with_scVI_latent_representation_n_{n_latent}.csv")
    latent_df.to_csv(out_path)
    print(f"Saved latent representation (n_latent={n_latent}) to {out_path}")

## Reload and validate latent embeddings from disk (optional)

Skip this if you just ran the training cells above in the same session --
`adata.obsm` already has the embeddings. Use this instead to resume after a
restart, using only the CSVs saved to `output_dir`, without retraining.

In [ ]:
for n_latent in n_latent_values:
    file_path = os.path.join(output_dir, f"hvg_obsm_with_scVI_latent_representation_n_{n_latent}.csv")
    latent_df = pd.read_csv(file_path, index_col=0)

    # ensure indices match between the AnnData object and the latent representation
    if not latent_df.index.equals(adata.obs.index):
        raise ValueError(f"Index mismatch between latent representation (n={n_latent}) and AnnData object")

    latent_key = f"{latent_key_prefix}{n_latent}"
    adata.obsm[latent_key] = latent_df.values
    print(f"Added latent representation for n_latent={n_latent} to .obsm['{latent_key}']")

## Save integrated output

In [ ]:
adata.X = adata.layers["raw_counts"].copy()
if "log1p" in adata.uns:
    del adata.uns["log1p"]

adata.write(output_path)
adata